In [0]:
# Databricks Notebook: Customer_Account_Silver
# Cell 1: Customer Quarantine & SCD Type 2 Implementation

from pyspark.sql.functions import col, current_timestamp, lit, to_date, to_json, struct
from delta.tables import DeltaTable

# 1. Read raw Customer data from Bronze
df_raw_customer = spark.table("bankingpoc.bronze.customer")

# 2. Enforce Data Quality Rules (Email Regex)
email_regex = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
df_evaluated = df_raw_customer.withColumn("is_valid_email", col("email").rlike(email_regex))

# 3. Quarantine Invalid Records
df_quarantine = df_evaluated.filter(~col("is_valid_email")).select(
    lit("customer").alias("source_table"),
    col("customer_id").cast("string").alias("record_identifier"),
    lit("Malformed email address format").alias("rejection_reason"),
    to_json(struct([col(c) for c in df_raw_customer.columns])).alias("raw_record_json"),
    current_timestamp().alias("quarantined_timestamp")
)

if df_quarantine.count() > 0:
    df_quarantine.write.format("delta").mode("append").saveAsTable("bankingpoc.silver.quarantine_records")
    print(f"Quarantined {df_quarantine.count()} invalid customer row(s).")

# 4. Prepare Cleaned Staging Records
df_source_clean = df_evaluated.filter(col("is_valid_email")).select(
    col("customer_id").cast("int"),
    col("first_name").cast("string"),
    col("last_name").cast("string"),
    col("email").cast("string"),
    col("phone").cast("string"),
    to_date(col("date_of_birth")).alias("date_of_birth"),
    col("kyc_status").cast("string")
)

target_customer_table = "bankingpoc.silver.customer"

# 5. Initialize or Merge SCD Type 2
if not spark.catalog.tableExists(target_customer_table):
    (df_source_clean
        .withColumn("effective_start_date", current_timestamp())
        .withColumn("effective_end_date", lit(None).cast("timestamp"))
        .withColumn("is_current", lit(True))
        .write.format("delta")
        .mode("overwrite")
        .saveAsTable(target_customer_table))
    print(f"Initialized SCD2 table: {target_customer_table}")
else:
    target_delta = DeltaTable.forName(spark, target_customer_table)
    target_df = target_delta.toDF()

    # Find active records whose attributes have changed
    staged_updates = (df_source_clean.alias("src")
        .join(target_df.filter("is_current = true").alias("tgt"), "customer_id")
        .filter(
            (col("src.first_name") != col("tgt.first_name")) |
            (col("src.last_name") != col("tgt.last_name")) |
            (col("src.email") != col("tgt.email")) |
            (col("src.phone") != col("tgt.phone")) |
            (col("src.kyc_status") != col("tgt.kyc_status"))
        )
        .select("src.*"))

    # Step A: Close out the existing active version for modified records
    target_delta.alias("tgt").merge(
        source=staged_updates.alias("stg"),
        condition="tgt.customer_id = stg.customer_id AND tgt.is_current = true"
    ).whenMatchedUpdate(set={
        "is_current": lit(False),
        "effective_end_date": current_timestamp()
    }).execute()

    # Step B: Insert both completely new customers and updated record versions
    target_delta.alias("tgt").merge(
        source=df_source_clean.alias("src"),
        condition="tgt.customer_id = src.customer_id AND tgt.is_current = true"
    ).whenNotMatchedInsert(values={
        "customer_id": "src.customer_id",
        "first_name": "src.first_name",
        "last_name": "src.last_name",
        "email": "src.email",
        "phone": "src.phone",
        "date_of_birth": "src.date_of_birth",
        "kyc_status": "src.kyc_status",
        "effective_start_date": "current_timestamp()",
        "effective_end_date": "null",
        "is_current": "true"
    }).execute()

    print("SCD Type 2 processing complete for Customer.")

Quarantined 1 invalid customer row(s).
SCD Type 2 processing complete for Customer.


In [0]:
# Cell 2: Account Cleansing & Merge

from pyspark.sql.functions import col, current_timestamp, to_date
from delta.tables import DeltaTable

df_raw_account = spark.table("bankingpoc.bronze.account")

# Validate balance integrity and normalize types
df_clean_account = df_raw_account.select(
    col("account_id").cast("int"),
    col("customer_id").cast("int"),
    col("branch_id").cast("int"),
    col("account_type").cast("string"),
    col("balance").cast("decimal(18,2)"),
    (col("balance") >= 0).alias("is_balance_valid"),
    col("account_status").cast("string"),
    to_date(col("opened_date")).alias("opened_date"),
    current_timestamp().alias("silver_processed_timestamp")
)

target_account = "bankingpoc.silver.account"

if not spark.catalog.tableExists(target_account):
    df_clean_account.write.format("delta").mode("overwrite").saveAsTable(target_account)
    print(f"Initialized table: {target_account}")
else:
    DeltaTable.forName(spark, target_account).alias("tgt").merge(
        source=df_clean_account.alias("src"),
        condition="tgt.account_id = src.account_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("Account upsert completed successfully.")

Account upsert completed successfully.
